In [1]:
import torch
import pickle
import os , re
import numpy as np
import librosa
import torch.optim as optim
from torch.utils.data import  Dataset , DataLoader
path = "../data/new/agent_0"
file = os.listdir(path= path)[0]
file_path = os.path.join(path, file)
file_path

'../data/new/agent_0/offline_episode_0_1633.pkl'

In [2]:
with open(file_path , 'rb') as f:
    data = pickle.load(f)
data[0].keys()

dict_keys(['camera', 'audio', 'step', 'rl_pred', 'rl_logits', 'rl_value', 'reward', 'mask', 'lstm_h', 'lstm_c'])

In [3]:
camera = data[0]['camera']
audio = data[0]['audio']
step = data[0]['step']
print(camera.shape)
print(audio.shape)
print(step.shape)

(61, 128, 128, 4)
(61, 2, 18000)
(61,)


In [6]:
mel_features = []
for i in range(audio.shape[0]):
    left_channel = audio[i, 0, :]
    right_channel = audio[i, 1, :]
    
    mel_left = librosa.feature.melspectrogram(y=left_channel, sr=18000, n_fft=1024, hop_length=512, n_mels=128)
    mel_right = librosa.feature.melspectrogram(y=right_channel, sr=18000, n_fft=1024, hop_length=512, n_mels=128)
    combined_mel = np.stack([mel_left, mel_right], axis=0)  # (2, 128, 时间帧数)
    mel_features.append(combined_mel)

In [13]:
a = np.array(mel_features)

In [8]:
import torch
import torch.nn as nn
# 1200 , sample , 
# 可以直接进行方向特征的计算
class Network(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden_dim = 128
        self.x_dim = 36
        self.y_dim = 128
        self.output_dim = 4
        
        self.audiomask = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * self.x_dim * self.y_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 2*self.x_dim * self.y_dim),
            nn.Sigmoid()
        )
        
        self.audio_encoder = nn.Sequential(
            nn.Conv2d(2, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 32 * 9, self.hidden_dim)
        )
        
        self.visual_encoder = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 32 * 32, self.hidden_dim)
        )
        
        self.fc1 = nn.Linear(2 * self.hidden_dim, self.hidden_dim)
        self.fc2 = nn.Linear(self.hidden_dim, self.output_dim)
    
    def forward(self, audio, visual_input):
        mel_features = []
        for i in range(audio.shape[0]):
            left_channel = audio[i, 0, :]
            right_channel = audio[i, 1, :]
            
            mel_left = librosa.feature.melspectrogram(y=left_channel, sr=18000, n_fft=1024, hop_length=512, n_mels=128)
            mel_right = librosa.feature.melspectrogram(y=right_channel, sr=18000, n_fft=1024, hop_length=512, n_mels=128)
            combined_mel = np.stack([mel_left, mel_right], axis=0)  # (2, 128, 时间帧数)
            mel_features.append(combined_mel)

        audio_input= np.array(mel_features)  # (61, 2, 128, 时间帧数)
        audio_input = torch.from_numpy(audio_input)
        visual_input = torch.from_numpy(visual_input)
        visual_input = visual_input.permute(0, 3, 1, 2)
        
        audio_mask = self.audiomask(audio_input)
        audio_mask = audio_mask.view(audio_input.size(0), 2, self.y_dim, self.x_dim)
        masked_audio = audio_mask * audio_input
        
        audio_encode = self.audio_encoder(masked_audio)
        visual_encode = self.visual_encoder(visual_input)
        
        combined_encode = torch.cat((audio_encode, visual_encode), dim=1)
        combined_encode = self.fc1(combined_encode)
        av_out = self.fc2(combined_encode)
        
        return av_out


In [18]:
type(a)

numpy.ndarray

In [9]:
net = Network()
print(net.state_dict())

OrderedDict([('audiomask.0.weight', tensor([[[[-6.9562e-02, -1.3793e-01, -3.1455e-02],
          [-1.3636e-01,  2.2540e-01,  1.7964e-01],
          [ 1.6683e-01, -1.4659e-01,  1.8300e-01]],

         [[ 1.4369e-01, -1.5078e-01, -9.8386e-02],
          [ 2.2793e-01, -8.0760e-02,  5.5574e-02],
          [-7.9856e-02,  1.7866e-01, -1.9653e-01]]],


        [[[-4.8251e-02, -5.3487e-02,  2.1626e-01],
          [-7.5348e-02,  7.9018e-02,  1.0958e-01],
          [ 2.1310e-01,  2.2279e-01,  3.3643e-02]],

         [[-1.5161e-01, -1.7542e-01, -1.1440e-02],
          [ 4.2647e-02, -5.2918e-02,  1.5105e-01],
          [-1.7660e-01,  2.3194e-01, -1.5491e-01]]],


        [[[-1.0388e-01,  1.3600e-01,  2.2382e-01],
          [ 1.7299e-01, -5.9787e-04, -5.7878e-02],
          [-3.7377e-02, -5.8720e-02,  1.5223e-01]],

         [[-3.3925e-02,  3.4384e-02, -1.4307e-01],
          [-7.6763e-02,  7.5804e-02, -1.6273e-01],
          [-1.1500e-01, -1.9972e-01, -1.7569e-01]]],


        [[[-1.4771e-01,  7.6

In [85]:
# one-hot

# 使用one-hot

# 监督学习

# 蒸馏， 大推小。

In [4]:
data[0]['audio'][:10].shape

(10, 2, 18000)

In [26]:
class MyData(Dataset):
    def __init__(self, data):
        self.data = data
        self.index_3 = np.where(self.data[0]['rl_pred'] == 3)[0].item()
        self.audio = list()
        self.visual = list()
        self.pred_tag = list()
        for data in self.data:
            self.audio.append(data['audio'][:self.index_3])
            self.visual.append(data['camera'][:self.index_3])
            self.pred_tag.append(data['rl_pred'][:self.index_3])

        self.audio = torch.from_numpy(np.array(self.audio))
        self.visual = torch.from_numpy(np.array(self.visual))
        self.pred_tag = torch.from_numpy(np.array(self.pred_tag))

        # # Flatten the data so that each sample is an individual entry
        self.audio = self.audio.view(-1, *self.audio.shape[2:])
        self.visual = self.visual.view(-1, *self.visual.shape[2:])
        self.pred_tag = self.pred_tag.view(-1, *self.pred_tag.shape[2:])

    def __len__(self):
        return len(self.audio)

    def __getitem__(self, idx):
        # Return a sample from audio, visual and pred_tag
        return self.audio[idx], self.visual[idx], self.pred_tag[idx]

    # def collate_fn(self, batch):
    #     # Create four lists to store samples for each label (0, 1, 2, 3)
    #     batch_0 = []
    #     batch_1 = []
    #     batch_2 = []
    #     batch_3 = []

    #     # Split batch based on the pred_tag labels
    #     for sample in batch:
    #         audio, visual, label = sample
    #         if label == 0:
    #             batch_0.append((audio, visual, label))
    #         elif label == 1:
    #             batch_1.append((audio, visual, label))
    #         elif label == 2:
    #             batch_2.append((audio, visual, label))
    #         elif label == 3:
    #             batch_3.append((audio, visual, label))

    #     # Ensure all four batches have the same number of samples (get the minimum size)
    #     min_size = min(len(batch_0), len(batch_1), len(batch_2), len(batch_3))

    #     # If any batch is empty, skip this batch
    #     if min_size == 0:
    #         return None  # Indicate that this batch should be skipped

    #     # Select samples from each batch to form a balanced batch
    #     balanced_batch = []
    #     for _ in range(min_size):
    #         balanced_batch.append(batch_0.pop())
    #         balanced_batch.append(batch_1.pop())
    #         balanced_batch.append(batch_2.pop())
    #         balanced_batch.append(batch_3.pop())

    #     # Shuffle the balanced batch
    #     np.random.shuffle(balanced_batch)

    #     # Return the batch with balanced classes
    #     return torch.utils.data.dataloader.default_collate(balanced_batch)

In [27]:
# 定义设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Network()
dataset = MyData(data)
dataloader = DataLoader(dataset=dataset , batch_size=4 , shuffle=True)
num_epochs = 1
criterion = nn.CrossEntropyLoss()  
optimizer = optim.Adam(model.parameters(), lr=1e-3)  
for epoch in range(num_epochs):
    model.train()
    for batch_audio, batch_visual, batch_labels in dataloader:
    # batch_audio = data_['audio']
    # batch_visual = data_['visual']
    # batch_labels = torch.from_numpy(data_['pred'])
    # batch_audio = batch_audio
    # batch_visual = batch_visual
    # batch_labels = batch_labels  # 分类任务
    
        optimizer.zero_grad()
        print(batch_audio.shape)
        # outputs = model(batch_audio, batch_visual)
        
        # loss = criterion(outputs, batch_labels)
        # loss.backward()
        # optimizer.step()
        
    # print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}")


torch.Size([4, 2, 18000])
torch.Size([4, 2, 18000])
torch.Size([4, 2, 18000])
torch.Size([4, 2, 18000])
torch.Size([4, 2, 18000])


In [ ]:
network  = Network()
result = network(batch_audio,batch_visual)
result.shape

In [13]:
class MyData(Dataset):
    def __init__(self,data):
        self.data = data
        self.audio = list()
        self.visual = list()
        self.pred_tag = list()
        for data in self.data:
            self.audio.append(data['audio'][:10])
            self.visual.append(data['camera'][:10])
            self.pred_tag.append(data['rl_pred'][:10])
        self.audio = torch.from_numpy(np.array(self.audio))
        self.visual = torch.from_numpy(np.array(self.visual))
        self.pred_tag = torch.from_numpy(np.array(self.pred_tag))
        self.audio = self.audio.view(-1 , *self.audio.shape[2:])
        self.visual = self.visual.view(-1 , *self.visual.shape[2:])
        self.pred_tag = self.pred_tag.view(-1 , *self.pred_tag.shape[2:])
    def __len__(self):
        return len(self.audio)
    def __getitem__(self , idx):
        # 我们核心需要三个数据，audio  ， img ， pred_tag ， 并且取前十个
        return self.audio[idx] , self.visual[idx] , self.pred_tag[idx]

In [9]:
data[0].keys()

dict_keys(['camera', 'audio', 'step', 'rl_pred', 'rl_logits', 'rl_value', 'reward', 'mask', 'lstm_h', 'lstm_c'])

In [22]:
data[0].keys()
dataset = MyData(data)

In [23]:
dataset
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

In [26]:
a = list()
b = list()
c = list()
for audio , visual , pred_tag in dataloader:
    print(audio.shape)

torch.Size([16, 2, 18000])
torch.Size([16, 2, 18000])
torch.Size([16, 2, 18000])
torch.Size([16, 2, 18000])
torch.Size([16, 2, 18000])
torch.Size([16, 2, 18000])
torch.Size([16, 2, 18000])
torch.Size([16, 2, 18000])
torch.Size([16, 2, 18000])
torch.Size([16, 2, 18000])


In [17]:
a[0].shape

torch.Size([16, 2, 18000])

In [22]:
a[0].shape

torch.Size([16, 10, 2, 18000])

In [27]:
print(d[9].shape)
print(a[0][0][9].shape)

torch.Size([2, 18000])
torch.Size([2, 18000])


In [30]:
print(d[9])
print(a[0][0][9])
if d[9].equal(a[0][0][9]):
    print("yes")

tensor([[ 9.0350e-01,  1.0829e+00,  1.2224e+00,  ..., -4.9380e-02,
         -1.2314e-01, -2.0504e-01],
        [ 2.9090e-01,  2.1158e-01,  1.0400e-01,  ..., -3.9457e-02,
         -6.9514e-03, -9.7030e-04]])
tensor([[ 9.0350e-01,  1.0829e+00,  1.2224e+00,  ..., -4.9380e-02,
         -1.2314e-01, -2.0504e-01],
        [ 2.9090e-01,  2.1158e-01,  1.0400e-01,  ..., -3.9457e-02,
         -6.9514e-03, -9.7030e-04]])
yes


In [1]:
import torch
torch.cuda.is_available()

True